# Phase 3: Workforce Intelligence — Step 3.1: Engagement Analytics

This notebook performs descriptive aggregation and reporting on employee engagement using `data/processed/engagement_data.csv`. Since department data is part of the same analytics cohort, we pull the department mappings (`DepartmentType`) from the raw dataset `data/raw/hr_performance_engagement.csv` by matching `Employee ID`.

We analyze:
1. Overall average engagement, satisfaction, and work-life balance.
2. Engagement metrics aggregated by department.
3. A list of the lowest-engagement employee records to identify target groups for retention programs.

In [1]:
import os
import pandas as pd
import numpy as np

proc_dir = os.path.join("data", "processed")
raw_dir = os.path.join("data", "raw")
print(f"Processed directory: {os.path.abspath(proc_dir)}")

Processed directory: C:\Users\Harshit Mishra\OneDrive\Desktop\enterprise_hr_ai\data\processed


## 1. Load Datasets
We load the cleaned `engagement_data.csv` and retrieve the department mapping from the raw performance and engagement dataset.

In [2]:
df_eng = pd.read_csv(os.path.join(proc_dir, "engagement_data.csv"))
df_raw = pd.read_csv(os.path.join(raw_dir, "hr_performance_engagement.csv"))

print(f"Cleaned Engagement Data Shape: {df_eng.shape}")
print(f"Raw Engagement Data Shape: {df_raw.shape}")

Cleaned Engagement Data Shape: (2843, 5)
Raw Engagement Data Shape: (2845, 28)


## 2. Merge Department Information
We map `Employee ID` to `DepartmentType` to enable department-level aggregation.

In [3]:
dept_mapping = dict(zip(df_raw["Employee ID"], df_raw["DepartmentType"]))
df_eng["Department"] = df_eng["Employee ID"].map(dept_mapping)

print("Sample mapped data:")
print(df_eng.head())
print(f"\nMissing departments after mapping: {df_eng['Department'].isnull().sum()}")

Sample mapped data:
   Employee ID Survey Date  ...  Work-Life Balance Score         Department
0         3427  14-01-2023  ...                        3  Production       
1         3428  09-09-2022  ...                        5  Production       
2         3429  27-05-2023  ...                        1              Sales
3         3430  16-06-2023  ...                        4              Sales
4         3431  25-11-2022  ...                        3              Sales

[5 rows x 6 columns]

Missing departments after mapping: 0


## 3. Overall Average Metrics
We calculate the mean and standard deviation of Engagement, Satisfaction, and Work-Life Balance scores across the entire workforce.

In [4]:
overall_avg = df_eng[["Engagement Score", "Satisfaction Score", "Work-Life Balance Score"]].mean()
overall_std = df_eng[["Engagement Score", "Satisfaction Score", "Work-Life Balance Score"]].std()

print("=== Overall Workforce Averages ===")
for col in overall_avg.index:
    print(f"{col}: {overall_avg[col]:.4f} (std: {overall_std[col]:.4f})")

=== Overall Workforce Averages ===
Engagement Score: 2.9406 (std: 1.4351)
Satisfaction Score: 3.0292 (std: 1.4100)
Work-Life Balance Score: 2.9887 (std: 1.4087)


## 4. Engagement by Department
We group by `Department` and calculate average engagement, satisfaction, and work-life balance scores, sorted by engagement in descending order.

In [5]:
dept_stats = df_eng.groupby("Department")[["Engagement Score", "Satisfaction Score", "Work-Life Balance Score"]].agg(["mean", "count"])
dept_stats.columns = [f"{col[0]} ({col[1]})" for col in dept_stats.columns]
dept_stats = dept_stats.sort_values(by="Engagement Score (mean)", ascending=False)

print("=== Department Aggregated Engagement Stats ===")
print(dept_stats)

=== Department Aggregated Engagement Stats ===
                      Engagement Score (mean)  ...  Work-Life Balance Score (count)
Department                                     ...                                 
Executive Office                     3.375000  ...                               24
IT/IS                                3.024450  ...                              409
Sales                                2.983923  ...                              311
Software Engineering                 2.955357  ...                              112
Admin Offices                        2.949367  ...                               79
Production                           2.908805  ...                             1908

[6 rows x 6 columns]


## 5. Lowest Engagement Records
We identify the top 20 employee records with the lowest engagement scores.

In [6]:
lowest_eng = df_eng.sort_values(by=["Engagement Score", "Satisfaction Score"], ascending=True).head(20)

print("=== Top 20 Lowest Engagement Records ===")
print(lowest_eng.to_string(index=False))

=== Top 20 Lowest Engagement Records ===
 Employee ID Survey Date  Engagement Score  Satisfaction Score  Work-Life Balance Score        Department
        3601  01-09-2022                 1                   1                        2 Production       
        3616  04-08-2023                 1                   1                        1 Production       
        3627  29-01-2023                 1                   1                        4 Production       
        3667  18-11-2022                 1                   1                        2 Production       
        3691  07-12-2022                 1                   1                        5 Production       
        3695  28-08-2022                 1                   1                        4 Production       
        3715  20-05-2023                 1                   1                        4 Production       
        3718  05-10-2022                 1                   1                        3 Production       
     